# Computational Theory Problems

My work on the Computational Theory problems. They're all based on the [Secure Hash Standard, FIPS 180-4](https://doi.org/10.6028/NIST.FIPS.180-4), the NIST document that defines SHA-256.
When I write something like (section 4.2.2), it means that section of the standard.

## Problem 0: GitHub Issues

Progress is tracked in this repo's [Issues](https://github.com/qahramantaha/computational-theory/issues): one issue per problem, and issues are closed when each problem is done.

## Problem 1: Representing SHA-256 Data

### 32-bit words
A word is 32 bits (section 2.1) and when you add words you do it mod 2³² (section 3.2). Python ints can be any size, so lecturer suggested to use numpy's [`uint32`](https://numpy.org/doc/stable/reference/arrays.scalars.html#numpy.uint32) instead of using MASK. It's a fixed 32-bit type, so it wraps back to 0 on its own and I don't need to mask anything.

numpy gives a warning when it wraps, so I hide it with [`np.errstate`](https://numpy.org/doc/stable/reference/generated/numpy.errstate.html).

In [1]:
# Numerical arrays and fixed size types like uint32.
import numpy as np

# Biggest number a 32-bit word can be, all 32 bits are 1.
w = np.uint32(0xFFFFFFFF)

# Show.
print(w)

# Hide the overflow warning, the wrap is what we want here.
with np.errstate(over="ignore"):
    # Adding 1 wraps back to 0, same as mod 2^32 (section 3.2).
    print(w + np.uint32(1))

# A normal Python int just keeps going past 32 bits.
print(int(w) + 1)

4294967295
0
4294967296


### Sequences of 32-bit words
I use a numpy array with `dtype=np.uint32` ([numpy.array docs](https://numpy.org/doc/stable/reference/generated/numpy.array.html)). It keeps the order, I can index it like `W[t]`, and every item is guaranteed to be 32 bits.

To show it I use the SHA-256 initial hash value H(0) from section 5.3.3.

In [2]:
# SHA-256 initial hash value H(0), section 5.3.3.
H0 = np.array([0x6a09e667, 0xbb67ae85, 0x3c6ef372, 0xa54ff53a,
               0x510e527f, 0x9b05688c, 0x1f83d9ab, 0x5be0cd19], dtype=np.uint32)

# Number of words, should be 8.
print(len(H0))

# First word in hex, should be 0x6a09e667.
print(hex(H0[0]))

# Type of the items, should be uint32.
print(H0.dtype)

8
0x6a09e667
uint32


### Input messages
Messages are stored as [`bytes`](https://docs.python.org/3/library/stdtypes.html#bytes). Text gets turned into bytes with [`.encode("utf-8")`](https://docs.python.org/3/library/stdtypes.html#str.encode), and each byte is a number from 0 to 255. The standard's own example uses "abc" (section 5.1.1), so I use the same one to check.

In [3]:
# Turn the text into bytes using UTF-8.
msg = "abc".encode("utf-8")

# Show the bytes.
print(msg)

# Each byte as a number.
print(list(msg))

# Each byte in binary, should match section 5.1.1: 01100001 01100010 01100011.
print([f"{b:08b}" for b in msg])

# Length in bits, should be 24.
print(len(msg) * 8)

b'abc'
[97, 98, 99]
['01100001', '01100010', '01100011']
24


### 512-bit message blocks
A block is 512 bits, which is 64 bytes, and gets split into 16 words (sections 3.1 and 5.2.1). I just use 64 made-up bytes here to show the split. Real blocks come from padding, which gets done later.

Each group of 4 bytes is turned into a word with [`int.from_bytes`](https://docs.python.org/3/library/stdtypes.html#int.from_bytes), reading it big-endian.

In [4]:
# 64 made-up bytes (0 to 63), which is 512 bits.
block = bytes(range(64))

# Split the block into 16 words, 4 bytes each, read big-endian.
words = [int.from_bytes(block[i:i+4], "big") for i in range(0, 64, 4)]

# Size of the block in bits, should be 512.
print(len(block) * 8)

# Number of words, should be 16.
print(len(words))

# First 3 words in hex.
print([hex(x) for x in words[:3]])

512
16
['0x10203', '0x4050607', '0x8090a0b']


### Final 256-bit hash value
The hash is 8 words stuck together (section 6.2), so 8 × 32 = 256 bits. I keep it as 8 words while working, then turn each word into 4 bytes with [`int.to_bytes`](https://docs.python.org/3/library/stdtypes.html#int.to_bytes) and join them. I'm using H(0) here just as an example of 8 words.

In [5]:
# Turn each word into 4 big-endian bytes and join them, using H(0) as the example.
hash_bytes = b"".join(int(x).to_bytes(4, byteorder="big") for x in H0)

# Size in bits, should be 256.
print(len(hash_bytes) * 8)

# The 32 bytes as hex text.
print(hash_bytes.hex())

# Number of hex characters, should be 64.
print(len(hash_bytes.hex()))

256
6a09e667bb67ae853c6ef372a54ff53a510e527f9b05688c1f83d9ab5be0cd19
64


To compare, this is a real SHA-256 hash of "abc" from Python's [`hashlib`](https://docs.python.org/3/library/hashlib.html). It comes out the same size and format.

In [6]:
# Standard hash functions.
import hashlib

# Real SHA-256 of "abc" as raw bytes.
real = hashlib.sha256(b"abc").digest()

# Size in bits, also 256.
print(len(real) * 8)

# Show the real hash in hex.
print(real.hex())

256
ba7816bf8f01cfea414140de5dae2223b00361a396177a9cb410ff61f20015ad


### Big-endian integers
SHA-256 is big-endian (section 3.1), which means the most important byte comes first. Section 3.1 uses 291 as an example, which should be the word `00000123`. Python lets us pick the byte order, so I always pass `"big"`. Using `"little"` flips the bytes and gives a different number. This matters because the message length added in padding is stored big-endian too (section 5.1.1).

In [7]:
# 291 as big-endian, should be 00000123 like section 3.1.
print((291).to_bytes(4, "big").hex())

# 291 as little-endian, the bytes come out backwards.
print((291).to_bytes(4, "little").hex())

# Turn the hex from section 3.1 back into bytes.
b = bytes.fromhex("00000123")

# Read big-endian, gives 291 back.
print(int.from_bytes(b, "big"))

# Read little-endian, gives the wrong number 587268096.
print(int.from_bytes(b, "little"))

00000123
23010000
291
587268096


## Problem 2: SHA-256 Bitwise Operations

### Helpers: SHR and ROTR
Section 4.1.2 builds the Sigma functions out of two operations on 32-bit words from section 3.2, so I write them first.

- **SHR** (shift right) moves the bits right by `n`. Bits falling off the right are lost and zeros come in on the left.
- **ROTR** (rotate right) is the same, but bits falling off the right wrap back round to the left: `(x >> n) | (x << 32 - n)`.

Python's shift operators `>>` and `<<` are explained in the [Python docs](https://docs.python.org/3/library/stdtypes.html#bitwise-operations-on-integer-types).

In [8]:
def SHR(x, n):
    """Shift the 32-bit word x right by n bits, filling with zeros on the left (FIPS 180-4, 3.2)."""
    # Make sure x is a 32-bit word.
    x = np.uint32(x)
    # Shift right, bits that fall off the right are lost.
    return x >> np.uint32(n)


def ROTR(x, n):
    """Rotate the 32-bit word x right by n bits, so bits off the right wrap to the left (FIPS 180-4, 3.2)."""
    # Make sure x is a 32-bit word.
    x = np.uint32(x)
    # Shift right, then OR in the bits that fell off, moved round to the left.
    return (x >> np.uint32(n)) | (x << np.uint32(32 - n))

### Ch(x, y, z)
Ch means "choose" (section 4.1.2). For each bit, if x is 1 it takes the bit from y, and if x is 0 it takes the bit from z.

Ch(x, y, z) = (x AND y) XOR (NOT x AND z)

In [9]:
def Ch(x, y, z):
    """Choose: for each bit, take y where x is 1 and z where x is 0 (FIPS 180-4, 4.1.2)."""
    # Make sure all three are 32-bit words.
    x, y, z = np.uint32(x), np.uint32(y), np.uint32(z)
    # y where x is 1, z where x is 0.
    return (x & y) ^ (~x & z)

To test Ch. If x is all 1s it should give back y, if x is all 0s it should give back z, and if x is half and half it should give half of each.

In [10]:
# x is all 1s, so it takes y.
assert Ch(0xFFFFFFFF, 0x12345678, 0x9ABCDEF0) == 0x12345678

# x is all 0s, so it takes z.
assert Ch(0x00000000, 0x12345678, 0x9ABCDEF0) == 0x9ABCDEF0

# First half of x is 1s so it takes y there, second half takes z.
assert Ch(0xFFFF0000, 0xFF00FF00, 0xF0F0F0F0) == 0xFF00F0F0

# The answer is still a 32-bit word.
assert Ch(0xFFFFFFFF, 0, 0).dtype == np.uint32

# Only gets here if all the asserts passed.
print("Ch tests passed")

Ch tests passed


### Maj(x, y, z)
Maj means majority (section 4.1.2). For each bit it outputs whichever value appears at least twice across x, y and z.

Maj(x, y, z) = (x AND y) XOR (x AND z) XOR (y AND z)

In [11]:
def Maj(x, y, z):
    """Majority: each output bit is whatever at least two of x, y, z have in that position (FIPS 180-4, 4.1.2)."""
    # Make sure all three are 32-bit words.
    x, y, z = np.uint32(x), np.uint32(y), np.uint32(z)
    # A bit is 1 if any two of them have a 1 there.
    return (x & y) ^ (x & z) ^ (y & z)

To test Maj. Two words of all 1s should win, two words of all 0s should win and the mixed one is worked out one hex digit at a time.

In [12]:
# Two of them are all 1s, so 1s win.
assert Maj(0xFFFFFFFF, 0xFFFFFFFF, 0x00000000) == 0xFFFFFFFF

# Two of them are all 0s, so 0s win.
assert Maj(0x00000000, 0x00000000, 0xFFFFFFFF) == 0x00000000

# Majority in each hex digit.
assert Maj(0xFFFF0000, 0xFF00FF00, 0xF0F0F0F0) == 0xFFF0F000

# The answer is still a 32-bit word.
assert Maj(0xFFFFFFFF, 0, 0).dtype == np.uint32

# Only gets here if all the asserts passed.
print("Maj tests passed")

Maj tests passed


### Sigma0(x)
XORs three rotated copies of x (section 4.1.2):

Σ0(x) = ROTR 2(x) XOR ROTR 13(x) XOR ROTR 22(x)

In [13]:
def Sigma0(x):
    """Σ0: ROTR 2 XOR ROTR 13 XOR ROTR 22 of x (FIPS 180-4, 4.1.2)."""
    # XOR three rotated copies of x.
    return ROTR(x, 2) ^ ROTR(x, 13) ^ ROTR(x, 22)

To test, I use x = 1. Rotating 1 right by n puts the bit at position 32 - n so the answer can be worked out by hand: bits 30, 19 and 10.

In [14]:
# 1 rotated by 2, 13 and 22 puts bits at 30, 19 and 10.
assert Sigma0(1) == 0x40080400

# The answer is still a 32-bit word.
assert Sigma0(0xFFFFFFFF).dtype == np.uint32

# Only gets here if all the asserts passed.
print("Sigma0 tests passed")

Sigma0 tests passed


### Sigma1(x), Σ1 in the standard
Same as Sigma0 with different rotations (section 4.1.2):

Σ1(x) = ROTR 6(x) XOR ROTR 11(x) XOR ROTR 25(x)

In [15]:
def Sigma1(x):
    """Σ1: ROTR 6 XOR ROTR 11 XOR ROTR 25 of x (FIPS 180-4, 4.1.2)."""
    # XOR three rotated copies of x.
    return ROTR(x, 6) ^ ROTR(x, 11) ^ ROTR(x, 25)

Same test with x = 1. This time the bits end up at 26, 21 and 7.

In [16]:
# 1 rotated by 6, 11 and 25 puts bits at 26, 21 and 7.
assert Sigma1(1) == 0x04200080

# The answer is still a 32-bit word.
assert Sigma1(0xFFFFFFFF).dtype == np.uint32

# Only gets here if all the asserts passed.
print("Sigma1 tests passed")

Sigma1 tests passed


### sigma0(x), σ0 in the standard
Two rotates and one shift (section 4.1.2).

σ0(x) = ROTR 7(x) XOR ROTR 18(x) XOR SHR 3(x)

In [17]:
def sigma0(x):
    """σ0: ROTR 7 XOR ROTR 18 XOR SHR 3 of x (FIPS 180-4, 4.1.2)."""
    # XOR two rotated copies and one shifted copy of x.
    return ROTR(x, 7) ^ ROTR(x, 18) ^ SHR(x, 3)

SHR loses bits instead of wrapping them so I test with x = 1 (SHR loses the bit) and x = 0x80000000 (SHR keeps it).

In [18]:
# Bits at 25 and 14 from the rotates, SHR loses the bit.
assert sigma0(1) == 0x02004000

# Bits at 24 and 13 from the rotates, and 28 from SHR.
assert sigma0(0x80000000) == 0x11002000

# The answer is still a 32-bit word.
assert sigma0(0xFFFFFFFF).dtype == np.uint32

# Only gets here if all the asserts passed.
print("sigma0 tests passed")

sigma0 tests passed


### sigma1(x), σ1 in the standard
This is the same as sigma0 but with different amounts (section 4.1.2).

σ1(x) = ROTR 17(x) XOR ROTR 19(x) XOR SHR 10(x)

In [19]:
def sigma1(x):
    """σ1: ROTR 17 XOR ROTR 19 XOR SHR 10 of x (FIPS 180-4, 4.1.2)."""
    # XOR two rotated copies and one shifted copy of x.
    return ROTR(x, 17) ^ ROTR(x, 19) ^ SHR(x, 10)

Same two test values as sigma0.

In [20]:
# Bits at 15 and 13 from the rotates, SHR loses the bit.
assert sigma1(1) == 0x0000A000

# Bits at 14 and 12 from the rotates, and 21 from SHR.
assert sigma1(0x80000000) == 0x00205000

# The answer is still a 32-bit word.
assert sigma1(0xFFFFFFFF).dtype == np.uint32

# Only gets here if all the asserts passed.
print("sigma1 tests passed")

sigma1 tests passed


## Problem 3: Generating the SHA-256 Constants

SHA-256 uses 64 constant 32-bit words called K (section 4.2.2). They're the first 32 bits of the fractional parts of the cube roots of the first 64 primes. Here I work them out myself and then check them against the ones in the standard.

### Testing for primes
First I need primes. To test if a number is prime I use the square root test from the lecturer's [prime numbers notes](https://github.com/ianmcloughlin/computational-theory/blob/main/notes/prime-numbers.ipynb). You only need to try dividing by numbers up to the square root of n, because if n has a factor bigger than that it must also have one smaller than it.

In [21]:
# Square root test for primality, from the prime numbers notes.
def isprime(n):
    """Test whether n is prime."""
    # Loop through 2...floor(sqrt(n)).
    for i in range(2, np.floor(np.sqrt(n)).astype(int) + 1):
        # If n divides evenly by i, then n is not prime.
        if n % i == 0:
            return False
    # If we get here, then n is prime.
    return True

To test it I use the same numbers as the notes. 10 and 17 × 19 aren't prime, 97 is.

In [22]:
# 10 is 2 x 5, so not prime.
assert isprime(10) == False

# 17 x 19 is not prime.
assert isprime(17 * 19) == False

# 97 is prime.
assert isprime(97) == True

# Only gets here if all the asserts passed.
print("isprime tests passed")

isprime tests passed


### primes(n)
Now count up from 2 and keep every number that passes `isprime` until there are n of them. It's the same idea as `find_primes` in the notes, except n gets passed in instead of always being 10.

In [23]:
def primes(n):
    """Return a list of the first n prime numbers."""
    # Start with an empty list.
    found = []
    # Start with 2.
    i = 2
    # Keep going until we have n primes.
    while len(found) < n:
        # If i is prime.
        if isprime(i):
            # Add it to the list.
            found.append(i)
        # Move to the next number.
        i += 1
    # Return the list.
    return found

The first 10 primes are 2 up to 29. I need 64 of them, and the 64th prime is 311.

In [24]:
# The first 10 primes.
assert primes(10) == [2, 3, 5, 7, 11, 13, 17, 19, 23, 29]

# The first 64 primes, used for the rest of this problem.
first64 = primes(64)

# There should be 64 of them and the last one should be 311.
assert len(first64) == 64
assert first64[-1] == 311

# Show them.
print(first64)

[2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47, 53, 59, 61, 67, 71, 73, 79, 83, 89, 97, 101, 103, 107, 109, 113, 127, 131, 137, 139, 149, 151, 157, 163, 167, 173, 179, 181, 191, 193, 197, 199, 211, 223, 227, 229, 233, 239, 241, 251, 257, 263, 269, 271, 277, 281, 283, 293, 307, 311]


### Cube roots
A cube root is the power of a third, the same way a square root is the power of a half (from the [roots notes](https://github.com/ianmcloughlin/computational-theory/blob/main/notes/roots.ipynb)). Python's [power operator](https://docs.python.org/3/reference/expressions.html#the-power-operator) `**` does it. I put the primes in a numpy array first so all 64 get done in one line.

In [25]:
# Cube root of each of the first 64 primes, as floats.
roots = np.array(first64) ** (1/3)

# Show the first 4.
print(roots[:4])

[1.25992105 1.44224957 1.70997595 1.91293118]


To check, cubing the roots should give back the primes. Floats aren't exact, so I use [`np.isclose`](https://numpy.org/doc/stable/reference/generated/numpy.isclose.html) like the roots notes do.

In [26]:
# Cubing each root should give back its prime, close enough for floats.
assert np.all(np.isclose(roots ** 3, first64))

# Only gets here if the assert passed.
print("cube root test passed")

cube root test passed


### Fractional parts
The fractional part is whatever is after the decimal point. [`np.floor`](https://numpy.org/doc/stable/reference/generated/numpy.floor.html) gives the whole number part so taking that away leaves the fraction. For example the cube root of 2 is 1.2599..., so the fractional part is 0.2599...

In [27]:
# Take away the whole number part to leave the fraction.
fracs = roots - np.floor(roots)

# Show the first 4.
print(fracs[:4])

[0.25992105 0.44224957 0.70997595 0.91293118]


Every fractional part should be at least 0 and less than 1.

In [28]:
# Every fraction is between 0 and 1.
assert np.all(fracs >= 0)
assert np.all(fracs < 1)

# Only gets here if the asserts passed.
print("fractional part test passed")

fractional part test passed


### 32-bit unsigned integers
To get the first 32 bits of the fraction I multiply it by 2³². That moves those 32 bits in front of the decimal point. Then [`astype(np.uint32)`](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.astype.html) drops whatever is left after the point and stores each one as a 32-bit word. The fraction is always less than 1 so the answer always fits in 32 bits.

In [29]:
# Move the first 32 bits of the fraction in front of the point, then keep the whole number part as a uint32.
K = (fracs * 2**32).astype(np.uint32)

# There should be 64 words.
assert len(K) == 64

# They should all be 32-bit words.
assert K.dtype == np.uint32

# Only gets here if the asserts passed.
print("uint32 test passed")

# They should all be 32-bit words.
assert K.dtype == np.uint32

# Only gets here if the asserts passed.
print("uint32 test passed")

uint32 test passed
uint32 test passed


### Constants in hexadecimal
Printed in hex, 8 per row, the same layout as section 4.2.2 so they're easy to compare by eye.

In [30]:
# Go through the constants 8 at a time.
for i in range(0, 64, 8):
    # Print each one as 8 hex digits, with a space between them.
    print(" ".join(f"{k:08x}" for k in K[i:i+8]))

428a2f98 71374491 b5c0fbcf e9b5dba5 3956c25b 59f111f1 923f82a4 ab1c5ed5
d807aa98 12835b01 243185be 550c7dc3 72be5d74 80deb1fe 9bdc06a7 c19bf174
e49b69c1 efbe4786 0fc19dc6 240ca1cc 2de92c6f 4a7484aa 5cb0a9dc 76f988da
983e5152 a831c66d b00327c8 bf597fc7 c6e00bf3 d5a79147 06ca6351 14292967
27b70a85 2e1b2138 4d2c6dfc 53380d13 650a7354 766a0abb 81c2c92e 92722c85
a2bfe8a1 a81a664b c24b8b70 c76c51a3 d192e819 d6990624 f40e3585 106aa070
19a4c116 1e376c08 2748774c 34b0bcb5 391c0cb3 4ed8aa4a 5b9cca4f 682e6ff3
748f82ee 78a5636f 84c87814 8cc70208 90befffa a4506ceb bef9a3f7 c67178f2


### Comparing with the standard
These are the 64 constants copied from section 4.2.2. I check every one of mine matches using [`np.array_equal`](https://numpy.org/doc/stable/reference/generated/numpy.array_equal.html).

In [31]:
# The 64 constants from section 4.2.2 of the standard.
K_standard = np.array([
    0x428a2f98, 0x71374491, 0xb5c0fbcf, 0xe9b5dba5, 0x3956c25b, 0x59f111f1, 0x923f82a4, 0xab1c5ed5,
    0xd807aa98, 0x12835b01, 0x243185be, 0x550c7dc3, 0x72be5d74, 0x80deb1fe, 0x9bdc06a7, 0xc19bf174,
    0xe49b69c1, 0xefbe4786, 0x0fc19dc6, 0x240ca1cc, 0x2de92c6f, 0x4a7484aa, 0x5cb0a9dc, 0x76f988da,
    0x983e5152, 0xa831c66d, 0xb00327c8, 0xbf597fc7, 0xc6e00bf3, 0xd5a79147, 0x06ca6351, 0x14292967,
    0x27b70a85, 0x2e1b2138, 0x4d2c6dfc, 0x53380d13, 0x650a7354, 0x766a0abb, 0x81c2c92e, 0x92722c85,
    0xa2bfe8a1, 0xa81a664b, 0xc24b8b70, 0xc76c51a3, 0xd192e819, 0xd6990624, 0xf40e3585, 0x106aa070,
    0x19a4c116, 0x1e376c08, 0x2748774c, 0x34b0bcb5, 0x391c0cb3, 0x4ed8aa4a, 0x5b9cca4f, 0x682e6ff3,
    0x748f82ee, 0x78a5636f, 0x84c87814, 0x8cc70208, 0x90befffa, 0xa4506ceb, 0xbef9a3f7, 0xc67178f2,
], dtype=np.uint32)

In [32]:
# Show the first and last from both, mine and the standard.
print(f"K[0]  = {K[0]:08x}   standard = {K_standard[0]:08x}")
print(f"K[63] = {K[63]:08x}   standard = {K_standard[63]:08x}")

# Every one of my constants matches the standard.
assert np.array_equal(K, K_standard)

# Only gets here if they all match.
print("All 64 constants match the standard")

K[0]  = 428a2f98   standard = 428a2f98
K[63] = c67178f2   standard = c67178f2
All 64 constants match the standard
